# Runde 1b · Gesamtauswertung auf CPU · H_tq-Variante

**In Konto A öffnen**, in einer frischen Laufzeit. GPU-Notebooks benötigen eine A100; ANA verwendet CPU.

**Post hoc, Änderung 2:** kleinere operative Neuronenmenge H_tq; Fragen, Stichproben, Schlüssel und Schwellen bleiben unverändert. Die bisherige Restschätzung von 15–18 A100-Stunden ist eine alte Planbasis, keine Laufzeitzusage; Umstellung und nötige Neuberechnungen kommen hinzu.

**Voraussetzung:** `HTQ_START` wurde erfolgreich abgeschlossen. Dieses Notebook prüft die Variantenordner vor dem Kopieren und Starten. K_S4 wird einschließlich des früher mit H+ gerechneten K8b neu gerechnet; alte K/L-Ausgaben und Caches werden nicht als H_tq-Ergebnisse weiterverwendet.

**Eine GPU:** START → B2_in_A → B1_in_A → A1_DC_zuerst → A2 → ANA. L-CAL separat auf CPU mit hohem RAM; `l_cal.json` vor ANA übernehmen. Danach die finanzierten Zusatzläufe und den Zusatzbericht abschließen. Höchstens zwei Laufzeiten je Konto gleichzeitig.

**Ausführen:** Laufzeittyp einstellen, dann „Alle ausführen“. Bei Abbruch dasselbe H_tq-Notebook erneut ausführen; erfolgreiche Variantenschritte werden übersprungen. Ergebnisse liegen unter `dsse_runs/variants/htq_v2`.


In [ ]:
T0 = __import__('time').time()
SLOT, KONTO = "ANA", "A"
PILOT = False             # Pilot-Slot (vor dem Einfrieren)
BRANCH = "codex/r1b-htq-continuation"
SELBSTTEST = False           # MOCK-Selbsttest dieses Slots vor dem Hauptlauf (einige Minuten; einmal je Code-Stand)
NUR_SELBSTTEST = False       # True: nur einrichten und testen, kein Hauptlauf
WARTEN_MAX_H = 10            # so lange wartet eine Sitzung höchstens auf ihre Startbedingungen
ABFRAGE_S = 120              # Abstand der Drive-Abfragen in Sekunden
LAUFZEIT_MAX_H = 23          # keine Sitzung startet, deren geplante Dauer über diese Laufzeit hinausreicht
import os, sys, json, shutil, subprocess, hashlib, lzma, time
print("CPU-Laufzeit (SG2: r1b_ana läuft auf einer erklärten CPU-Laufzeit).")

In [ ]:
# Getrennte Variantenordner; originale Ergebnisordner werden nur gelesen.
from google.colab import drive
drive.mount("/content/drive")
MYDRIVE = "/content/drive/MyDrive"
SOURCE_RUNS = f"{MYDRIVE}/dsse_runs"
assert os.path.isdir(SOURCE_RUNS), "Der originale dsse_runs-Ordner fehlt."
SOURCE_A = SOURCE_RUNS
SOURCE_B = f"{MYDRIVE}/dsse_runs_B"
assert os.path.isdir(SOURCE_B), "Verknüpfung dsse_runs_B fehlt."
RUNS = os.path.join(SOURCE_RUNS, "variants/htq_v2")
WORK, ROOT, REPO = "/content/work_htq_v2", "/content/root", "/content/DSSE"
os.makedirs(WORK, exist_ok=True); os.makedirs(ROOT, exist_ok=True)
# Nur für reale Unterprozesse; MOCK-Selbsttests erhalten keine Drive-/Variantenvariablen.
_variant_keys = ("R1B_H_POLICY", "R1B_RUNS_B", "R1B_XFER_A", "R1B_SOURCE_RUNS", "R1B_CAPTURE_HIDDEN")
TEST_ENV = {k: v for k, v in os.environ.items() if k not in _variant_keys}
VARIANT_ENV = dict(TEST_ENV, R1B_H_POLICY="htq_v2")
B_RUNS = os.path.join(SOURCE_B, "variants/htq_v2")
LINK = f"{SOURCE_A}/r1b_xfer"
VARIANT_ENV.update(R1B_RUNS_B=B_RUNS, R1B_XFER_A=LINK)
print("H_tq-Ergebnisse:", RUNS)
print("Unveränderte R_S0-Exporte:", LINK)


In [ ]:
# Code: analysis/ju_space (mit round1b/ und round2/), der Plan und die Runde-2-Ergebnisse; ohne --depth, denn der Runner prüft
# den Einfrier-Commit (SG1) und den Code der Runde-1-Teile an ihren Commits (SG3)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "transformers==5.22.2"], check=True)
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--sparse", "-b", BRANCH, "https://github.com/Erikiss/DSSE", REPO], check=True)
subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", "analysis/ju_space", "results/ju_space/plans",
                "results/ju_space/runde2_ergebnisse"], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %cd %s"], capture_output=True, text=True).stdout.strip())
A = f"{REPO}/analysis/ju_space"; R1B = f"{A}/round1b"; D = f"{A}/gpu/data"
FZ_PATH = f"{REPO}/results/ju_space/plans/runde1b/runde1b_freeze.json"
PINS_PATH = f"{REPO}/results/ju_space/plans/runde1b/runde1b_pilot_pins.json"    # vor dem Einfrieren: Pins für die Piloten
FZ = json.load(open(FZ_PATH)) if os.path.exists(FZ_PATH) else {}
if not FZ and PILOT and os.path.exists(PINS_PATH):
    FZ = json.load(open(PINS_PATH))
print("Einfrier-Datensatz:", "vorhanden" if os.path.exists(FZ_PATH) else ("fehlt; Pilot-Pins (Modelle, Datensätze, J-Lenses) aus runde1b_pilot_pins.json"
      if FZ else "fehlt (nur die Piloten r1b_P0 / r1b_P0b laufen vor dem Einfrieren)"))

In [ ]:
# Varianten-Bootstrap / Prüfung vor jedem realen Kopier- oder Hauptlauf.
# Variantenwerkzeug: Ausgabe und Fehler mit Konto sichtbar streamen.
import textwrap
VARIANT_TOOL = f"{R1B}/r1b_htq_variant.py"
def run_variant(action, path, account):
    def show(message):
        for line in str(message).splitlines() or [""]:
            for wrapped in textwrap.wrap(line, width=72, replace_whitespace=False,
                                         drop_whitespace=False, break_on_hyphens=False) or [""]:
                print(wrapped, flush=True)
    show(f"[Konto {account}] {action}: {path}")
    command = [sys.executable, "-u", VARIANT_TOOL, action, path, account]
    try:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, env=TEST_ENV)
    except OSError as error:
        show(f"[Konto {account}] {action}: Start fehlgeschlagen: {error}")
        raise RuntimeError(f"Konto {account}: {action} konnte nicht starten.") from error
    try:
        for line in process.stdout:
            show(line.rstrip("\r\n"))
        code = process.wait()
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        show(f"[Konto {account}] {action}: unterbrochen; Kindprozess beendet.")
        raise
    finally:
        process.stdout.close()
    show(f"[Konto {account}] {action}: Exit {code}")
    if code:
        raise RuntimeError(f"Konto {account}: {action} fehlgeschlagen (Exit {code}). Details oben.")
    return code

def check_variant(path, account):
    return run_variant("check", path, account)
if not NUR_SELBSTTEST:
    check_variant(RUNS, KONTO)
    check_variant(B_RUNS, "B")
    pair = json.load(open(os.path.join(RUNS, "_htq_pair_ready.json")))
    assert pair.get("variant") == "htq_v2" and pair.get("accounts") == ["A", "B"], "HTQ_START muss beide Konten vorbereiten."
    done = os.path.join(B_RUNS, "r1b_K_S1", "session_done.json")
    partial = os.path.join(B_RUNS, "r1b_K_S1", "session_partial.json")
    assert os.path.isfile(done) and not os.path.exists(partial), "Zuerst HTQ_START bis zum fertigen neuen K_S1 ausführen."


In [ ]:
# Eingaben: Modelle zur festen Revision mit Prüfsummen (r1b_core.ensure_model; models/verified.json), für Teil R die Persona-Sätze
# und Runde-1-Daten (wie Runde 2), die J-Lens (Prüfsumme festgelegt); Datensätze lädt jeder Schritt selbst zur festen Revision.
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()

MODELLE = []
if MODELLE and not NUR_SELBSTTEST:
    subprocess.run([sys.executable, f"{R1B}/r1b_core.py", "fetch-models", ROOT] + MODELLE, check=True)
if PILOT and not NUR_SELBSTTEST:              # Pilot: PG19-Pilotbücher (GCS, sha256 je Datei) und WikiText-103 (Hugging Face) vorab
    subprocess.run([sys.executable, f"{R1B}/r1b_core.py", "fetch-datasets", ROOT, "pg19", "wikitext103"], check=False)
if SELBSTTEST:                                # der MOCK-Selbsttest braucht nur Konfiguration und Tokenizer von Qwen3-1.7B
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3-1.7B", revision="70d244cc86ccca08cf5af4e1e306ecf908b1ad5e", local_dir=f"{ROOT}/models/qwen3-1.7b",
                      allow_patterns=["*.json", "merges.txt", "vocab.json"])
if False:                                # rk_loop-Pfade von Teil R (r2_rk.open_block): Persona-Sätze und Runde-1-Daten
    from huggingface_hub import hf_hub_download
    SUMS = json.load(open(f"{D}/checksums.json")); P = f"{ROOT}/rsi_dl/persona"; os.makedirs(P, exist_ok=True)
    for n, h in SUMS["persona_git_sha1"].items():
        data = open(hf_hub_download(SUMS["persona_repo"], f"persona/{n}.jsonl", repo_type="dataset", revision=SUMS["persona_revision"]), "rb").read()
        assert hashlib.sha1(b"blob %d\0" % len(data) + data).hexdigest() == h, f"Prüfsumme stimmt nicht: {n}"
        open(f"{P}/{n}.jsonl", "wb").write(data)
    for src, dst in SUMS["inputs"].items():
        os.makedirs(os.path.dirname(f"{ROOT}/{dst['path']}"), exist_ok=True)
        raw = lzma.open(f"{D}/{src}").read() if src.endswith(".xz") else open(f"{D}/{src}", "rb").read()
        open(f"{ROOT}/{dst['path']}", "wb").write(raw)
        assert sha256(f"{ROOT}/{dst['path']}") == dst["sha256"], f"Prüfsumme stimmt nicht: {dst['path']}"
if False and not NUR_SELBSTTEST:          # SO-JLENS: aus dsse_runs/r1b_inputs (einmal je Konto hochladen) oder von der URL im Einfrier-Datensatz
    PIN = "6fcc79011bd921ffd87612255e2e99950a124fa519470ee44ebaf161c39be9d6"
    ZIEL = f"{ROOT}/jlens/Qwen3-1.7B_jacobian_lens.pt"; os.makedirs(os.path.dirname(ZIEL), exist_ok=True)
    QUELLE = f"{RUNS}/r1b_inputs/Qwen3-1.7B_jacobian_lens.pt"
    if not os.path.exists(ZIEL) and os.path.exists(QUELLE):
        shutil.copyfile(QUELLE, ZIEL)
    elif not os.path.exists(ZIEL) and (FZ.get("jlens") or {}).get("url"):
        import urllib.request; urllib.request.urlretrieve(FZ["jlens"]["url"], ZIEL)
    if not os.path.exists(ZIEL):                # sonst direkt von Hugging Face (Neuronpedia), dieselbe Datei, Prüfsumme unten
        from huggingface_hub import hf_hub_download
        shutil.copyfile(hf_hub_download("neuronpedia/jacobian-lens", "qwen3-1.7b/jlens/Salesforce-wikitext/Qwen3-1.7B_jacobian_lens.pt"), ZIEL)
    if os.path.exists(ZIEL):
        assert sha256(ZIEL) == PIN, "J-Lens: Prüfsumme stimmt nicht (SO-JLENS)"
        print("J-Lens geprüft.")
    else:
        print("J-Lens FEHLT: dsse_runs/r1b_inputs/Qwen3-1.7B_jacobian_lens.pt hochladen; Schritte, die sie brauchen, scheitern sonst.")
if False and not NUR_SELBSTTEST:   # Teil K (K5a, RK-2): J-Lens von Qwen3.5-2B (Name, sha256: Einfrier-Datensatz jlens.qwen3.5-2b)
    J2 = (FZ.get("jlens") or {}).get("qwen3.5-2b") or {}
    if J2.get("name"):
        ZIEL2, QUELLE2 = f"{ROOT}/jlens/{J2['name']}", f"{RUNS}/r1b_inputs/{J2['name']}"
        os.makedirs(os.path.dirname(ZIEL2), exist_ok=True)
        if not os.path.exists(ZIEL2) and os.path.exists(QUELLE2):
            shutil.copyfile(QUELLE2, ZIEL2)
        elif not os.path.exists(ZIEL2) and J2.get("url"):      # sonst von der URL im Einfrier-Datensatz (Hugging Face, Neuronpedia)
            import urllib.request; urllib.request.urlretrieve(J2["url"], ZIEL2 + ".part"); os.replace(ZIEL2 + ".part", ZIEL2)
        if not os.path.exists(ZIEL2):
            from huggingface_hub import hf_hub_download
            shutil.copyfile(hf_hub_download("neuronpedia/jacobian-lens", "qwen3.5-2b/jlens/Salesforce-wikitext/" + J2["name"]), ZIEL2)
        OK2 = os.path.exists(ZIEL2) and sha256(ZIEL2) == J2.get("sha256")
        if os.path.exists(ZIEL2) and not OK2:
            os.remove(ZIEL2)                            # falsche Prüfsumme: Datei verwerfen
        print("J-Lens Qwen3.5-2B:", "geprüft" if OK2 else "FEHLT oder Prüfsumme falsch (K5a / RK-2 lesen dann 'not run')")
print("Eingaben bereit:", ", ".join(MODELLE) or "keine Modelle nötig")

In [ ]:
# Zweiter Ordner: was die Sitzungen dieses Slots aus dem anderen Konto lesen (Konto A: fertige Ordner aus dsse_runs_B; Konto B:
# r1b_xfer_A); der Runner wiederholt das bei jeder Abfrage, diese Zelle zeigt den Stand vorab
subprocess.run([sys.executable, f"{R1B}/r1b_run_slot.py", SLOT, ROOT, WORK, RUNS, "--copy-only"], env=dict(VARIANT_ENV, R1B_ACCOUNT=KONTO))

In [ ]:
# Selbsttest (nur Code): alle Sitzungen dieses Slots und ihre Vorgänger im MOCK-Modus auf einem Schein-Drive (zwei Konten, Verknüpfungen
# als symbolische Links), über denselben Runner wie der Hauptlauf; nichts geht ins echte Drive. Marke je Code-Stand in dsse_runs.
MARKE = f"{RUNS}/r1b_selftest_{SLOT}_{COMMIT[:12]}.json"
if not SELBSTTEST:
    print("Selbsttest ausgelassen (SELBSTTEST = False).")
elif os.path.exists(MARKE):
    print("Selbsttest für diesen Code-Stand schon bestanden:", json.load(open(MARKE)))
else:
    t = time.time(); ST = "/content/r1b_selftest"; shutil.rmtree(ST, ignore_errors=True)
    r = subprocess.run([sys.executable, "-u", f"{R1B}/r1b_run_slot.py", SLOT, ROOT, ST, "--selftest"],
                       env=dict(TEST_ENV, RUN_DEVICE="cpu", NT="4"))
    if r.returncode:
        raise SystemExit(f"Selbsttest fehlgeschlagen (Protokolle unter {ST}/work/*/*/logs).")
    json.dump(dict(commit=COMMIT, seconds=round(time.time() - t)), open(MARKE, "w"))
    print(f"Selbsttest bestanden ({time.time() - t:.0f}s).")

In [ ]:
# Hauptlauf: die Sitzungen des Slots nacheinander; jede mit SG1-SG7, Startbedingungen, Kopie der Eingaben, Manifest, Schritten mit
# Zeitgrenze und Drive-Abgleich
MELDUNG = {0: "Alle Sitzungen dieses Slots sind fertig oder 'not run' (Stufe 2 nicht finanziert / nicht ausgelöst).",
           1: "Eine Sitzung ist zweimal gescheitert (siehe oben); Protokolle prüfen, ggf. als 'not run' erklären, Notebook erneut ausführen.",
           2: "Eine Freigabe (SG1, SG2, SG3 oder SG7) ist nicht erfüllt: manifest.json der Sitzung ansehen; diese Sitzung lief nicht.",
           3: "Laufzeitgrenze erreicht: Notebook in einer neuen Laufzeit erneut ausführen (Fertiges wird übersprungen).",
           4: "Startbedingungen nach WARTEN_MAX_H Stunden nicht erfüllt: später erneut ausführen."}
RC = None
if NUR_SELBSTTEST:
    print("NUR_SELBSTTEST = True: kein Hauptlauf.")
else:
    env = dict(VARIANT_ENV, R1B_ACCOUNT=KONTO, R1B_ROOT=ROOT, RUN_DEVICE="cpu", NT="8", R1B_RUNTIME_T0=repr(T0),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")   # weniger Speicherzersplitterung bei langen Kontexten
    cmd = [sys.executable, "-u", f"{R1B}/r1b_run_slot.py", SLOT, ROOT, WORK, RUNS, "--poll-s", str(ABFRAGE_S),
           "--wait-h", str(WARTEN_MAX_H), "--runtime-h", str(LAUFZEIT_MAX_H)]
    RC = subprocess.run(cmd, env=env).returncode
    print(MELDUNG.get(RC, f"Unerwarteter Rückgabewert {RC}"))

In [ ]:
# Übersicht: Zustand jeder Sitzung des Slots in Drive
for s in ['r1b_ana', 'r1b_ana2']:
    base = f"{RUNS}/{s}"
    for name in ("session_done.json", "session_partial.json", "session_not_run.json", "start_check.json"):
        if os.path.exists(f"{base}/{name}"):
            S = json.load(open(f"{base}/{name}")); print(f"{s}: {name}", S.get("reason") or "")
            for k, v in (S.get("steps") or {}).items():
                sec = f"{v['seconds']:.0f}s" if isinstance(v.get("seconds"), (int, float)) else ""
                print(f"    {k:<28} {v['state']:<9} {sec:>6}  {(v.get('note') or '')[:80]}")
            break
    else:
        print(f"{s}: noch nicht begonnen")

In [ ]:
# Abschluss: Drive schreiben und trennen
drive.flush_and_unmount()
print("Fertig. Ergebnisse in Google Drive unter dsse_runs/variants/htq_v2/<Sitzung>.")
if RC not in (None, 0):
    raise SystemExit(MELDUNG.get(RC, f"Rückgabewert {RC}"))